# Week 7 — Observability, privacy, reliability, cost, and GenAIOps

Trace model and tool behavior with OpenTelemetry/Application Insights while minimizing customer data. A release includes its model, prompt, tools, knowledge assets, evaluation evidence, operational limits, and rollback target.

In [ ]:
import importlib.util
import sys
from pathlib import Path

curriculum_root = next(
    candidate
    for base in (Path.cwd(), *Path.cwd().parents)
    for candidate in (base, base / "examples" / "foundry-curriculum")
    if (candidate / "notebook_setup.py").is_file()
)
spec = importlib.util.spec_from_file_location(
    "foundry_curriculum_setup", curriculum_root / "notebook_setup.py"
)
helpers = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = helpers
spec.loader.exec_module(helpers)
session = helpers.load_session(curriculum_root)
labs = helpers.load_offline_labs(curriculum_root)
session.safe_summary()

In [ ]:
operations_contract = {
    "slos": {
        "task_success_minimum": 0.85,
        "availability_minimum": 0.995,
        "p95_latency_seconds_maximum": 8.0,
        "cost_per_success_budget": "set-from-approved-pricing-source",
    },
    "trace_policy": {
        "prompt_capture": "off-by-default",
        "tool_argument_capture": "allow-listed-fields-only",
        "retention_days": "set-by-data-policy",
        "access_group": "group:ai-platform-operators",
    },
    "failure_drills": [
        "model throttling",
        "search unavailable",
        "tool timeout",
        "model version retirement",
    ],
    "degraded_mode": "answer only from approved static guidance",
}
operations_contract

In [ ]:
release_manifest = {
    "foundry_project": session.project_endpoint,
    "logical_model": session.logical_model,
    "model_deployment": session.deployment,
    "model_version": "change-v2-offline-fixture",
    "prompt_digest": "record-before-release",
    "tool_schema_versions": [],
    "knowledge_index_version": "record-before-release",
    "embedding_and_chunking_versions": "record-before-release",
    "evaluation_run_id": None,
    "red_team_result_id": None,
    "rollback_target": "baseline-v1",
    "evidence_source": labs.EVIDENCE_SOURCE,
}
release_manifest

In [ ]:
release_router = labs.SimulatedReleaseRouter(
    {"baseline-v1", "change-v2"}, active_version="change-v2"
)
healthy_result = release_router.invoke(
    "Explain project endpoints",
    correlation_id="trace-healthy",
    dependency_state="healthy",
)
throttled_result = release_router.invoke(
    "Explain project endpoints",
    correlation_id="trace-throttled",
    dependency_state="throttled",
)
unavailable_result = release_router.invoke(
    "Explain project endpoints",
    correlation_id="trace-unavailable",
    dependency_state="unavailable",
)
assert healthy_result.mode == "normal"
assert throttled_result.mode == unavailable_result.mode == "degraded"
assert "static guidance" in throttled_result.response

In [ ]:
allowed_trace_fields = {
    "correlation_id",
    "release",
    "dependency_state",
    "outcome",
    "latency_ms",
    "error_type",
    "query_digest",
    "evidence_source",
}
assert set(throttled_result.trace) == allowed_trace_fields
assert {"prompt", "query", "output", "tool_arguments"}.isdisjoint(
    throttled_result.trace
)

In [ ]:
rollback_result = release_router.rollback(release_manifest["rollback_target"])
post_rollback = release_router.invoke(
    "Explain project endpoints",
    correlation_id="trace-rollback",
    dependency_state="healthy",
)
assert rollback_result["status"] == "completed"
assert post_rollback.trace["release"] == "baseline-v1"
operations_evidence = {
    "healthy": healthy_result,
    "throttled": throttled_result,
    "unavailable": unavailable_result,
    "rollback": rollback_result,
}
operations_evidence

## Exit criteria

Demonstrate a throttled or failed dependency, verify degraded behavior, query latency/errors without exposing prompt data, and test rollback to a known-good immutable version. Do not assume Hosted Agent weighted traffic splitting is available.